# Punctuation restoration: BERT and DeBERTa-v3 on a Colab GPU

This notebook fine-tunes two pretrained encoders for the project's token classification task
(one label per word: `O`, `COMMA`, `PERIOD`, `QUESTION`) on the FOMC press conference data:

- **BERT** (`bert-base-uncased`), a vanilla method from the course slides (baseline);
- **DeBERTa-v3** (`microsoft/deberta-v3-base`), a newer model.

All training logic lives in `fomcpunct/transformer.py` (tested locally); this notebook only calls it.

**Protocol.** Learning rates are chosen with seed 0 on the validation split; the chosen rate is
then trained with seeds 1 and 2. Each run keeps its best epoch by validation macro-F1. Test
predictions are written for every run but are **not scored here**; all models are scored on the
test split once, at the end of the project, with `python -m fomcpunct.evaluate --split test`.

**Grid extended (23 October 2026).** The first search (BERT 2e-5, 3e-5, 5e-5; DeBERTa-v3 1e-5,
2e-5, 3e-5) picked the largest rate for both models, so one larger rate was added to each grid
(BERT 8e-5, DeBERTa-v3 5e-5). Rerunning the cells skips every finished run.

**Before you start**

1. *Runtime → Change runtime type → T4 GPU* (the project's runs used an A100; any GPU works).
2. Upload `p2_colab_bundle.zip` to Google Drive in the folder `MyDrive/coe025_p2/`.
3. Run the cells in order and keep the browser tab open (free Colab stops idle sessions).

**Drive space:** about 25 MB in total (about 1.3 MB per run). No model weights are stored: when
validation improves, the run writes that epoch's validation and test predictions instead.

**If Colab disconnects or a cell fails:** if the runtime is still connected, first use
*Runtime → Restart session*; then run the cells from the top again. Finished runs are skipped; the
run that was cut off starts again from the beginning (at most about 15 minutes lost).

**If you upload a new bundle:** first rename the folders `MyDrive/coe025_p2/runs` and
`MyDrive/coe025_p2/results` (for example to `runs_old`), then use *Runtime → Restart session*.

Expected time on a T4: about 5 minutes per BERT run and 10–15 minutes per DeBERTa run;
10–14 runs in total, so roughly 1.5–2.5 hours (an A100 is about three times faster).

In [ ]:
# 1. Check the GPU
import torch
!nvidia-smi --query-gpu=name,memory.total --format=csv
assert torch.cuda.is_available(), "No GPU: Runtime -> Change runtime type -> T4 GPU"
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Install the pinned packages (torch comes with Colab)
!pip -q install transformers==5.20.2 sentencepiece==0.2.2

In [ ]:
# 3. Mount Drive, unpack the bundle and check its file hashes
import hashlib, json, os, sys, zipfile
from pathlib import Path
from google.colab import drive

if "fomcpunct" in sys.modules:
    raise RuntimeError("fomcpunct is already loaded in this session: Runtime -> Restart session, then run from the top")

drive.mount("/content/drive")
WORKDIR = Path("/content/drive/MyDrive/coe025_p2")   # run folders and results stay here (a few MB)
BUNDLE = WORKDIR / "p2_colab_bundle.zip"
CODE = Path("/content/p2")                           # code and data, unpacked locally
assert BUNDLE.exists(), f"Upload p2_colab_bundle.zip to {WORKDIR}"

with zipfile.ZipFile(BUNDLE) as archive:
    archive.extractall(CODE)
manifest = json.loads((CODE / "BUNDLE_MANIFEST.json").read_text())
bad = [name for name, digest in manifest["files"].items()
       if hashlib.sha256((CODE / name).read_bytes()).hexdigest() != digest]
assert not bad, f"files changed in the bundle: {bad}"
print(f"bundle OK: {len(manifest['files'])} files, created {manifest['created_utc']}")
code_hash = hashlib.sha256(b"".join((CODE / "fomcpunct" / n).read_bytes()
                                    for n in ("transformer.py", "metrics.py", "__init__.py"))).hexdigest()[:16]
print("transformer code hash:", code_hash)

os.chdir(CODE)
sys.path.insert(0, str(CODE))

In [ ]:
# 4. Settings
from fomcpunct.transformer import CODE_HASH, MODELS, Config, export, select_lr, train_run
assert CODE_HASH == code_hash

DATA = CODE / "data" / "processed"
CFG = Config()            # max_len 256, overlap 64, batch 16, 4 epochs, warm-up 10%, fp16
# Grid extended on 23 October 2026 (8e-5 for BERT, 5e-5 for DeBERTa-v3): the first search picked the largest rate.
LEARNING_RATES = {"bert": [2e-5, 3e-5, 5e-5, 8e-5], "deberta": [1e-5, 2e-5, 3e-5, 5e-5]}
SEEDS = [0, 1, 2]
print(CFG)
print(MODELS)   # Hub models with pinned revisions

## BERT

Seed 0 with each learning rate, then the best rate with seeds 1 and 2.

In [ ]:
# 5. BERT: learning-rate search with seed 0
for lr in LEARNING_RATES["bert"]:
    train_run("bert", lr, 0, DATA, WORKDIR, CFG)

In [ ]:
# 6. BERT: chosen rate with seeds 1 and 2, then copy the outputs to results/
best_lr = select_lr("bert", LEARNING_RATES["bert"], WORKDIR)
print("chosen learning rate:", best_lr)
for seed in SEEDS[1:]:
    train_run("bert", best_lr, seed, DATA, WORKDIR, CFG)
export("bert", best_lr, SEEDS, WORKDIR)

## DeBERTa-v3

Same procedure. You may see a warning that the `deberta-v3` tokenizer has "an incorrect regex
pattern". It is a false positive for this model: locally, the tokenizer gave exactly the same
pieces as the original sentencepiece model for all 3,443 word types of the validation split.

In [ ]:
# 7. DeBERTa-v3: learning-rate search with seed 0
for lr in LEARNING_RATES["deberta"]:
    train_run("deberta", lr, 0, DATA, WORKDIR, CFG)

In [ ]:
# 8. DeBERTa-v3: chosen rate with seeds 1 and 2, then copy the outputs to results/
best_lr = select_lr("deberta", LEARNING_RATES["deberta"], WORKDIR)
print("chosen learning rate:", best_lr)
for seed in SEEDS[1:]:
    train_run("deberta", best_lr, seed, DATA, WORKDIR, CFG)
export("deberta", best_lr, SEEDS, WORKDIR)

In [ ]:
# 9. Pack the results (predictions, cards, learning curves, tuning, latency) for download
import shutil
from google.colab import files

archive = shutil.make_archive(str(WORKDIR / "p2_transformer_results"), "zip", WORKDIR, "results")
print("wrote", archive)
files.download(archive)

## After the runs

Send back `p2_transformer_results.zip` (it is also in `MyDrive/coe025_p2/`). In the repository:

```
python -m fomcpunct.colab import p2_transformer_results.zip
python -m fomcpunct.evaluate --split validation --models heuristic hmm crf bilstm_crf_s0 bilstm_crf_s1 bilstm_crf_s2 bert_s0 bert_s1 bert_s2 deberta_s0 deberta_s1 deberta_s2
```

The import step checks that every run was made from the same data and the same transformer code
as the repository (the code hash printed in cell 3). Only at the very end of the project, once
every model choice is fixed, the same `evaluate` command is run once with `--split test`.